# 15 — Train DeepSeek-V2 (phase 4)

## Two training tracks

| Track | Where | What |
|-------|--------|------|
| **A — PyTorch (full model)** | This notebook | `Trainer` updates MLA + MoE + embeddings (recommended) |
| **B — C demo (head only)** | `c/bin/train_v2_tiny` | Forward full model; SGD on tied `wte`/`lm_head` only |

Full MoE+MLA **backward in C** is Phase 5 (like all of `train_gpt2.c`).


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text, train_val_split
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config
from llmc.train import Trainer, TrainConfig

text = load_text("data/tiny_shakespeare.txt")
train_text, val_text = train_val_split(text)
tok = CharTokenizer.from_text(text)
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)

cfg = DeepSeekV2Config.tiny(tok.vocab_size, block_size=64)
model = DeepSeekV2(cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("device:", device, "| params:", f"{model.count_parameters():,}")

trainer = Trainer(
    model, train_ids, val_ids,
    TrainConfig(max_steps=200, batch_size=32, eval_interval=50, learning_rate=3e-3),
    device=device,
)


### A — PyTorch training (uncomment)


In [ ]:
# history = trainer.train()
# print("last val loss:", history[-1]["val"])
print("Uncomment history = trainer.train() for real training.")


### B — C trainer (forward + optional head SGD)

```bash
cd c
make bin/train_v2_tiny
./bin/train_v2_tiny              # 10 steps, print cross-entropy
./bin/train_v2_tiny -train-head 40   # demo: loss on lm_head only
```

Read `deepseek_v2/train_v2_tiny.c` next to `llmc/train.py` — same loop shape as llm.c.


In [ ]:
import shutil, subprocess
if shutil.which("make"):
    r = subprocess.run(["make", "-s", "bin/train_v2_tiny"], cwd="c", capture_output=True, text=True)
    if r.returncode == 0:
        r2 = subprocess.run(["./bin/train_v2_tiny"], cwd="c", capture_output=True, text=True)
        print(r2.stdout[-800:] if len(r2.stdout) > 800 else r2.stdout)


**Next:** notebook **16** — sample in PyTorch, then export weights for C.
